In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk import WordNetLemmatizer
import string

In [2]:
root_path = "/home/stefan/ioai-prep/kits/essay-gap"

# Dataset

In [3]:
stop_words = set(stopwords.words("english"))
to_remove = stop_words | set(string.punctuation)

lemmatizer = WordNetLemmatizer()

def clean_text(text: str):
    text = text.lower()
    tok = nltk.word_tokenize(text)
    tok = [lemmatizer.lemmatize(t) for t in tok if t not in to_remove]
    return ' '.join(tok)

clean_text("Ana has apples.")

'ana apple'

In [4]:
train_df = pd.read_csv(f"{root_path}/train.csv")

for col in train_df.select_dtypes(include='object'):
    train_df[col] = train_df[col].apply(clean_text)

train_df.head()

,sampleID,before,after,opt_0,opt_1,opt_2,opt_3,label
0,0,china nationalist communist force resumed civi...,middle east arab rejection united nation parti...,dog domesticated member family canidae,ad 493 book song quote frontier general tan da...,communist force prevailed established people '...,dead position neither player able checkmate le...,2
1,1,structure distribution coral reef charles darw...,subsidence continues fringing reef becomes bar...,fringing reef form around extinct volcanic isl...,coral larva settle sand build existing reef co...,popular motown recording late 1960s 1970s reli...,may remembered phrase `` white right '' `` que...,0
2,2,coral reef restoration grown prominence past s...,deterioration global reef fish nursery biodive...,following hindenburg 's death 1934 hitler proc...,coral stressor include pollution warming ocean...,violin tuned fifth note g3 d4 a4 e5,roosevelt reinforced philippine american prote...,1
3,3,world war ii transformed political economic so...,many country whose industry damaged moved towa...,walter chandoha made career photographing cat ...,python 2.5 possible pas data back generator fu...,jon krakauer 's thin air 1997 expressed author...,wake europe 's devastation influence great pow...,3
4,4,another important strategic question middlegam...,every reduction material good purpose example ...,male cat called tom tommy tomcat gib neutered,commonly cited reason using bitcoin include hi...,minor material advantage generally transformed...,however house cat behavior also influenced hum...,2


In [5]:
def common_words_cnt(text1: str, text2: str):
    words1 = set(text1.split())
    words2 = set(text2.split())
    return len(words1 & words2)

common_words_cnt("Ana has apples.", "Maria wants apples.")

1

# Submission

In [6]:
test_df = pd.read_csv(f"{root_path}/test.csv")

for col in train_df.select_dtypes(include="object"):
    test_df[col] = test_df[col].apply(clean_text)

In [7]:
answers = []

for idx in range(len(test_df)):
    row = test_df.iloc[idx]

    scores = [
        {
            "opt": i,
            "score": common_words_cnt(row["before"], row[f"opt_{i}"])
        } for i in range(4)
    ]

    ans = max(scores, key=lambda x: x["score"])["opt"]
    answers.append(ans)

In [8]:
submission = pd.DataFrame({"sampleID": test_df["sampleID"], "answer": answers})
submission.head()

,sampleID,answer
0,100,0
1,101,1
2,102,1
3,103,0
4,104,3


In [9]:
submission.to_csv("submission.csv", index=False)